# SRQ backend generalization — M3 FLY regression

This notebook checks that extracting the analytic backend did not change Exact FLY or the locked P2B SRQ-FLY implementation. It uses a CIFAR-100 **training-only** split, never materializes `test.pt`, performs no hyperparameter selection, and exports no feature/WTA cache. Run every cell in order on a T4 GPU.

In [ ]:
# Edit repository/path values only. Protocol and gate values are source-locked.
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH='experiment/soho-selfcontained'
WORK_DIR='/content/SOHO-CL'
FEATURE_CACHE_DIR='/content/srq_m3_cifar_features'
WTA_CACHE_DIR='/content/srq_m3_wta_10000'
OUTPUT_DIR='/content/srq_m3_output'
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
# Fresh clone, dependency install, GPU check, and immutable source verification.
import hashlib,json,os,shutil,subprocess,sys
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
EXPECTED={
 'configs/srq_generalization_m3_fly_regression.json':'d7f587c3a88e8bbc2165343b8dfa079e802c1be60dea2c26ea37b4798ae68e3e',
 'tools/srq_generalization_m3.py':'53d3cec06d3c97fde2e2922ccc57b9e3c58ed188ccb9bce5917c9811a7ef0a1e',
 'methods/analytic_ridge/backends.py':'e93817056e5666c86e5e6dd59343ab3108099ba41f53dc9866159b702a5a0aed',
 'methods/frontends/fly.py':'1576a98d9b422d68764d32c99e29e08971c4f456e6315968e7252397bdb480fa',
 'methods/srq_fly_optimized/learner.py':'40edac2e2cc88faac549f5c87217f3143d815bf53ecad8a37dfdb22c112691ae'}
for path,expected in EXPECTED.items(): assert sha(path)==expected,(path,sha(path),expected)
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip(),'Repository must start clean.'
CONFIG='configs/srq_generalization_m3_fly_regression.json'
RUNNER='tools/srq_generalization_m3.py'
print('GPU:',torch.cuda.get_device_name(0))
print('COMMIT:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('SOURCE LOCK: PASS')

In [ ]:
# Fast synthetic gate before any data download.
command=[sys.executable,'-m','pytest','-q','tests/test_srq_generalization_m3.py','tests/test_analytic_ridge_backend.py','tests/test_analytic_ridge_equivalence.py']
completed=subprocess.run(command)
assert completed.returncode==0,'Synthetic correctness gate failed; return the complete traceback.'
print('M3 SYNTHETIC GATE: PASS')

In [ ]:
# Download the locked backbone checkpoint and processed CIFAR-100 source.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CIFAR_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('CHECKPOINT:',CHECKPOINT_PATH)
print('CIFAR ROOT:',CIFAR_ROOT)

In [ ]:
# Materialize TRAIN features only. The held-out test tensor must stay absent.
cache=Path(FEATURE_CACHE_DIR)
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',CIFAR_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256','32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b','--feature-cache-dir',FEATURE_CACHE_DIR,'--output-dir','/content/unused_m3','--dataset','CIFAR-100','--model-name','vit_base_patch16_224','--data-augmentation','vit','--seed','2025','--num-classes','100','--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START',flush=True)
    subprocess.run(command,check=True)
assert (cache/'train.pt').is_file() and not (cache/'test.pt').exists()
metadata=json.loads((cache/'metadata.json').read_text())
assert metadata['feature_dim']==768 and metadata['finite'] is True
print('TRAIN CACHE READY:',metadata.get('train_shape'),'| test.pt absent')

In [ ]:
# Run the full ten-task train-only regression. WTA construction is resumable by cache identity.
command=[sys.executable,'-u',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',FEATURE_CACHE_DIR,'--code-cache-dir',WTA_CACHE_DIR,'--output-dir',OUTPUT_DIR,'--device','cuda','--require-clean-git']
print('M3 START: legacy/generic Exact FLY and P2B on one locked stream.',flush=True)
completed=subprocess.run(command)
result_path=Path(OUTPUT_DIR)/'m3_results.json'
assert result_path.is_file(),'M3 failed before writing diagnostics; return the complete runner output.'
result=json.loads(result_path.read_text())
print('STATUS:',result['status'])
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
assert completed.returncode==0 and result['status']=='PASS_M3_FLY_REGRESSION','M3 regression failed; do not relax gates.'

In [ ]:
# Human-readable task audit. Every identity flag must remain true.
import pandas as pd
rows=[]
for record in result['records']:
    for method in ('exact','p2b'):
        item=record[method]
        rows.append({'task':record['task'],'path':method,'state_bytes':item['generic_persistent_state_bytes'],'prediction_agreement':item['prediction_agreement'],'relative_logit_error':item['relative_logit_error'],'weights_identical':item['weights_identical']})
display(pd.DataFrame(rows))

In [ ]:
# Export evidence only. Sample-level feature and WTA caches are deliberately excluded.
bundle=Path('/content/srq_generalization_m3_fly_regression')
if bundle.exists(): shutil.rmtree(bundle)
bundle.mkdir()
shutil.copy2(Path(OUTPUT_DIR)/'m3_results.json',bundle/'m3_results.json')
shutil.copy2(CONFIG,bundle/'config.json')
archive=shutil.make_archive(str(bundle),'zip',root_dir=bundle)
print('ARTIFACT:',archive,'sha256=',sha(archive))
from google.colab import files
files.download(archive)